In [ ]:
!pip install torch ninja

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

# --- 1. CUDA KERNEL SOURCES ---
CUDA_V5_SRC = """
#include <torch/extension.h>
#include <cuda_runtime.h>
__global__ void sparse_dot_product_tiled_kernel(float* logits, const float* q, const float* k, const int64_t* topk_indices, const int B, const int H, const int T, const int D, const int K) {
    int b = blockIdx.z; int h = blockIdx.y; int t_query = blockIdx.x * blockDim.x + threadIdx.x;
    if (t_query < T) {
        extern __shared__ float s_query[];
        for (int d = 0; d < D; d++) { s_query[threadIdx.x * D + d] = q[((b * H + h) * T + t_query) * D + d]; }
        __syncthreads();
        for (int k_idx = 0; k_idx < K; k_idx++) {
            int t_key = topk_indices[b * T * K + t_query * K + k_idx];
            float score = 0.0f;
            for (int d = 0; d < D; d++) { score += s_query[threadIdx.x * D + d] * k[((b * H + h) * T + t_key) * D + d]; }
            logits[((b * H + h) * T + t_query) * K + k_idx] = score;
        }
    }
}
torch::Tensor sparse_attention_v5_tiled(torch::Tensor q, torch::Tensor k, torch::Tensor topk_indices) {
    auto B = q.size(0); auto H = q.size(1); auto T = q.size(2); auto D = q.size(3); auto K = topk_indices.size(2);
    auto logits = torch::empty({B, H, T, K}, q.options());
    const int threads = 128; const dim3 blocks((T + threads - 1) / threads, H, B);
    sparse_dot_product_tiled_kernel<<<blocks, threads, threads * D * sizeof(float)>>>(logits.data_ptr<float>(), q.data_ptr<float>(), k.data_ptr<float>(), topk_indices.data_ptr<int64_t>(), B, H, T, D, K);
    return logits;
}
"""

CUDA_V7_SRC = """
#include <torch/extension.h>
__global__ void fused_value_aggregation_v7_kernel(float* out, const float* attn, const float* v, const int64_t* topk, const int B, const int H, const int T, const int D, const int K) {
    int t_query = blockIdx.x * blockDim.x + threadIdx.x; int d = blockIdx.y * blockDim.y + threadIdx.y; int bh = blockIdx.z;
    if (t_query < T && d < D) {
        int b = bh / H; int h = bh % H; float sum = 0.0f;
        for (int k_idx = 0; k_idx < K; k_idx++) {
            int t_key = topk[b * T * K + t_query * K + k_idx];
            sum += attn[((b * H + h) * T + t_query) * K + k_idx] * v[((b * H + h) * T + t_key) * D + d];
        }
        out[((b * H + h) * T + t_query) * D + d] = sum;
    }
}
torch::Tensor sparse_value_aggregation_v7(torch::Tensor attn, torch::Tensor v, torch::Tensor topk) {
    auto B = v.size(0); auto H = v.size(1); auto T = v.size(2); auto D = v.size(3); auto K = topk.size(2);
    auto out = torch::empty({B, H, T, D}, v.options());
    dim3 threads(16, 16); dim3 blocks((T + threads.x - 1) / threads.x, (D + threads.y - 1) / threads.y, B * H);
    fused_value_aggregation_v7_kernel<<<blocks, threads>>>(out.data_ptr<float>(), attn.data_ptr<float>(), v.data_ptr<float>(), topk.data_ptr<int64_t>(), B, H, T, D, K);
    return out;
}
"""

# --- 2. JIT LOADING ---
print("Kompiliere SPA V7 Kernels...")
spa_ops_v5 = load_inline(name='spa_v5', cpp_sources='torch::Tensor sparse_attention_v5_tiled(torch::Tensor q, torch::Tensor k, torch::Tensor topk_indices);', cuda_sources=CUDA_V5_SRC, functions=['sparse_attention_v5_tiled'], with_cuda=True)
spa_ops_v7 = load_inline(name='spa_v7', cpp_sources='torch::Tensor sparse_value_aggregation_v7(torch::Tensor attn, torch::Tensor v, torch::Tensor topk);', cuda_sources=CUDA_V7_SRC, functions=['sparse_value_aggregation_v7'], with_cuda=True)

# --- 3. MODEL ARCHITECTURE ---
class SPA_V7_Model(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8, n_layers=4, k=24):
        super().__init__()
        self.k, self.n_heads, self.d_model = k, num_heads, embed_dim
        self.head_dim = embed_dim // num_heads
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(32768, embed_dim)
        self.layers = nn.ModuleList([nn.ModuleList([nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 3*embed_dim), nn.Linear(embed_dim, embed_dim), nn.LayerNorm(embed_dim), nn.Sequential(nn.Linear(embed_dim, 4*embed_dim), nn.GELU(), nn.Linear(4*embed_dim, embed_dim))]) for _ in range(n_layers)])
        self.ln_f, self.head = nn.LayerNorm(embed_dim), nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, tau_vals=None, tau_idx=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        if tau_idx is None:
            tau_idx = torch.randint(0, T, (B, T, self.k), device=idx.device, dtype=torch.long)
            tau_vals = torch.full((B, T, self.k), 0.01, device=idx.device)
        for ln1, qkv_l, proj, ln2, mlp in self.layers:
            h = ln1(x)
            qkv = qkv_l(h).reshape(B, T, 3, self.n_heads, self.head_dim)
            q, k, v = qkv[:,:,0].transpose(1,2).contiguous(), qkv[:,:,1].transpose(1,2).contiguous(), qkv[:,:,2].transpose(1,2).contiguous()
            logits = (spa_ops_v5.sparse_attention_v5_tiled(q, k, tau_idx) * (self.head_dim**-0.5)) + (5.0 * torch.log(tau_vals + 1e-8)).unsqueeze(1)
            attn = F.softmax(logits, dim=-1)
            out = spa_ops_v7.sparse_value_aggregation_v7(attn, v, tau_idx)
            x = x + proj(out.transpose(1, 2).reshape(B, T, self.d_model))
            x = x + mlp(ln2(x))
        return self.head(self.ln_f(x)), tau_vals, tau_idx

print("SPA V7 bereit fahrbereit!")

In [ ]:
import requests

# Download the tiny shakespeare dataset
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
r = requests.get(url)
with open('input.txt', 'w') as f:
    f.write(r.text)

print('Tiny Shakespeare dataset heruntergeladen.')

In [ ]:
# Read the dataset
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Create a vocabulary
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)

# Create a mapping from characters to integers and vice versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s): # encoder: take a string, output a list of integers
    return [stoi[c] for c in s]

def decode(l): # decoder: take a list of integers, output a string
    return ''.join([itos[i] for i in l])

# Convert the text to a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

print(f'Vokabulargröße: {VOCAB_SIZE}')
print(f'Datensatzgröße: {len(data)} Zeichen')
print(f'Beispiel (kodiert): {data[:20].tolist()}')
print(f'Beispiel (dekodiert): {decode(data[:20].tolist())}')

In [ ]:
# Instantiate the model
model = SPA_V7_Model(vocab_size=VOCAB_SIZE).cuda()

# Print the number of parameters
print(f'Modellparameter: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

In [ ]:
# --- 4. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3000)

print('Starte Training des skalierten Modells...')
model.train()
for step in range(3001):
    ix = torch.randint(len(data) - 256, (8,))
    xb = torch.stack([data[j:j+256] for j in ix]).cuda()
    yb = torch.stack([data[j+1:j+256+1] for j in ix]).cuda()

    logits, _, _ = model(xb)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if step % 500 == 0:
        print(f'Schritt {step} | Loss: {loss.item():.4f}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare a small context for attention visualization
viz_context = 'ROMEO: Wo bist du, Romeo?'
idx_viz = torch.tensor(encode(viz_context), dtype=torch.long).unsqueeze(0).cuda()
B_viz, T_viz = idx_viz.shape

# Set the model to evaluation mode
model.eval()

# Manually perform a forward pass up to the attention calculation for the last layer
with torch.no_grad():
    x_simulated = model.token_emb(idx_viz) + model.pos_emb(torch.arange(T_viz, device=idx_viz.device))

    # Get model parameters for attention calculation
    k_val = model.k
    n_heads_val = model.n_heads
    head_dim_val = model.head_dim

    last_layer_attn = None
    last_layer_tau_idx = None

    # Iterate through layers to capture attention from the last one
    for layer_idx, (ln1, qkv_l, proj, ln2, mlp) in enumerate(model.layers):
        h = ln1(x_simulated)
        qkv = qkv_l(h).reshape(B_viz, T_viz, 3, n_heads_val, head_dim_val)
        q, k_t, v = qkv[:,:,0].transpose(1,2).contiguous(), qkv[:,:,1].transpose(1,2).contiguous(), qkv[:,:,2].transpose(1,2).contiguous()

        # Generate tau_idx and tau_vals if not provided (as happens internally in the model)
        # For visualization, random sampling of top-K indices is fine.
        current_tau_idx = torch.randint(0, T_viz, (B_viz, T_viz, k_val), device=idx_viz.device, dtype=torch.long)
        current_tau_vals = torch.full((B_viz, T_viz, k_val), 0.01, device=idx_viz.device)

        logits = (spa_ops_v5.sparse_attention_v5_tiled(q, k_t, current_tau_idx) * (head_dim_val**-0.5)) + (5.0 * torch.log(current_tau_vals + 1e-8)).unsqueeze(1)
        current_attn = F.softmax(logits, dim=-1) # Shape (B, H, T, K)

        # Update x_simulated for the next layer
        out = spa_ops_v7.sparse_value_aggregation_v7(current_attn, v, current_tau_idx)
        x_simulated = x_simulated + proj(out.transpose(1, 2).reshape(B_viz, T_viz, model.d_model))
        x_simulated = x_simulated + mlp(ln2(x_simulated))

        # Store attention from the last layer
        if layer_idx == len(model.layers) - 1:
            last_layer_attn = current_attn
            last_layer_tau_idx = current_tau_idx

# Select the first batch item and first head for visualization
# Convert to numpy for plotting
attn_to_plot = last_layer_attn[0, 0].cpu().numpy() # Shape (T_viz, K)
tau_idx_to_plot = last_layer_tau_idx[0].cpu().numpy() # Shape (T_viz, K)

# Create a dense T_viz x T_viz attention matrix from sparse attention
dense_attn_matrix = np.zeros((T_viz, T_viz))
for i in range(T_viz): # Query position
    for j in range(k_val): # Top-K key index
        key_pos = tau_idx_to_plot[i, j]
        dense_attn_matrix[i, key_pos] = attn_to_plot[i, j]

print(f"Aufmerksamkeitsmatrix-Form für Heatmap: {dense_attn_matrix.shape}")

In [ ]:
# Decode tokens for axis labels
labels = [itos[i.item()] for i in idx_viz[0]]

plt.figure(figsize=(10, 8))
sns.heatmap(
    dense_attn_matrix, 
    cmap='viridis', 
    xticklabels=labels, 
    yticklabels=labels,
    linewidths=.5,
    linecolor='lightgray'
)
plt.xlabel('Schlüsselpositionen (Key Positions)')
plt.ylabel('Anfragepositionen (Query Positions)')
plt.title('Aufmerksamkeits-Heatmap (Letzte Schicht, Kopf 0)')
plt.tight_layout()
plt.show()

In [ ]:
# --- 5. INFERENCE ---
model.eval()
start_context = 'ROMEO: '
GEN_LENGTH = 500
T_CTX = 1024

idx = torch.tensor(encode(start_context), dtype=torch.long).unsqueeze(0).cuda()
print(f'Generiere {GEN_LENGTH} Tokens...')

with torch.no_grad():
    for _ in range(GEN_LENGTH):
        idx_cond = idx[:, -T_CTX:]
        logits, _, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :] / 0.7, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx), dim=1)

print('\n--- Ergebnis ---\n' + decode(idx[0].tolist()))

In [ ]:
# --- 6. ADVANCED INFERENCE (TOP-P / NUCLEUS SAMPLING) ---
def top_p_sampling(logits, p=0.9):
    # Sortiere Logits absteigend
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

    # Maske erstellen für Tokens, die außerhalb der Top-P liegen
    sorted_indices_to_remove = cumulative_probs > p
    # Verschiebe die Maske um eins nach rechts, um den ersten Token über P noch zu behalten
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0

    # Setze Logits der zu entfernenden Tokens auf -unendlich
    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    logits[indices_to_remove] = float('-inf')
    return logits

model.eval()
start_context = 'ROMEO: '
GEN_LENGTH = 500
T_CTX = 1024

# --- HYPERPARAMETER ---
TEMPERATURE = 0.7
TOP_P = 0.9  # Kumulative Wahrscheinlichkeits-Grenze

idx = torch.tensor(encode(start_context), dtype=torch.long).unsqueeze(0).cuda()
print(f'Generiere {GEN_LENGTH} Tokens (Temp: {TEMPERATURE}, Top-P: {TOP_P})...')

with torch.no_grad():
    for _ in range(GEN_LENGTH):
        idx_cond = idx[:, -T_CTX:]
        logits, _, _ = model(idx_cond)
        
        # Letzten Token isolieren
        logits = logits[0, -1, :] / TEMPERATURE
        
        # Top-P Filter anwenden
        filtered_logits = top_p_sampling(logits, p=TOP_P)
        
        # Sampling
        probs = F.softmax(filtered_logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx.unsqueeze(0)), dim=1)

print('\n--- Top-P Ergebnis ---\n' + decode(idx[0].tolist()))